In [2]:
from zeep import Client
from zeep.wsse.username import UsernameToken
import dotenv
import os
import datetime as dt
from zeep.exceptions import Fault
dotenv.load_dotenv()
token = os.environ.get("DARWIN_LITE_TOKEN")


In [3]:


try:
    response = client.service.GetDepartureBoard(numRows=10, crs="HDW")
    for service in response.trainServices.service:
        print(service)
except Fault as fault:
    print("SOAP Fault:", fault)

NameError: name 'client' is not defined

In [4]:
from zeep import Client, Settings, xsd
from zeep.plugins import HistoryPlugin

LDB_TOKEN = token
WSDL = 'http://lite.realtime.nationalrail.co.uk/OpenLDBWS/wsdl.aspx?ver=2021-11-01'

if LDB_TOKEN == '':
    raise Exception("Please configure your OpenLDBWS token in getDepartureBoardExample!")

settings = Settings(strict=False)

history = HistoryPlugin()

client = Client(wsdl=WSDL, settings=settings, plugins=[history])

header = xsd.Element(
    '{http://thalesgroup.com/RTTI/2013-11-28/Token/types}AccessToken',
    xsd.ComplexType([
        xsd.Element(
            '{http://thalesgroup.com/RTTI/2013-11-28/Token/types}TokenValue',
            xsd.String()),
    ])
)
header_value = header(TokenValue=LDB_TOKEN)

res = client.service.GetDepartureBoard(numRows=10, crs='HDW', _soapheaders=[header_value])

print("Trains at " + res.locationName)
print("===============================================================================")

services = res.trainServices.service

i = 0
while i < len(services):
    t = services[i]
    print(t.std + " to " + t.destination.location[0].locationName + " - " + "Platform: " + (t.platform if t.platform else "") + " - " + t.etd)
    i += 1

Trains at Hadley Wood
20:17 to Moorgate - Platform: 1 - On time
20:30 to Welwyn Garden City - Platform: 4 - On time
20:47 to Moorgate - Platform: 1 - On time
21:00 to Welwyn Garden City - Platform: 4 - On time
21:17 to Moorgate - Platform: 1 - On time
21:30 to Welwyn Garden City - Platform: 4 - On time
21:47 to Moorgate - Platform: 1 - On time
22:00 to Welwyn Garden City - Platform: 4 - On time


In [ ]:
type(services)

In [ ]:
def get_departure_board(crs_code: str, token: str, num_rows: int | None =10, filter_crs: str | None = None) -> list:

    WSDL = 'http://lite.realtime.nationalrail.co.uk/OpenLDBWS/wsdl.aspx?ver=2021-11-01'
    settings = Settings(strict=False)
    history = HistoryPlugin()
    client = Client(wsdl=WSDL, settings=settings, plugins=[history])

    header = xsd.Element(
        '{http://thalesgroup.com/RTTI/2013-11-28/Token/types}AccessToken',
        xsd.ComplexType([
            xsd.Element(
                '{http://thalesgroup.com/RTTI/2013-11-28/Token/types}TokenValue',
                xsd.String()),
        ])
    )
    header_value = header(TokenValue=LDB_TOKEN)

    try:
        res = client.service.GetDepartureBoard(numRows=num_rows, crs=crs_code, filterCrs=filter_crs, _soapheaders=[header_value])
    except Exception as e:
        print(f"Error fetching departure board: {e}")
        return []
    
    services = res.trainServices.service
    location_name = res.locationName
    generated = res.generatedAt
    return (location_name, generated, services)

In [ ]:
def print_train_info(services: list, location_name: str, timestamp: dt.datetime):
    print("Trains at " + location_name + " at " + timestamp.strftime("%Y-%m-%d %H:%M:%S"))
    print("===============================================================================")
    
    i = 0
    while i < len(services):
        t = services[i]
        print(t.std + " to " + t.destination.location[0].locationName + " - " + "Platform: " + (t.platform if t.platform else "") + " - " + t.etd)
        i += 1

In [ ]:
info = get_departure_board("MOG", token, 10, "HDW")
print_train_info(services=info[2], location_name=info[0], timestamp=info[1])

In [ ]:
res

In [ ]:
import re

pattern = re.compile(r"^train\s+[A-Za-z]{3}$", re.IGNORECASE)

pattern.match("train")